# Decision Trees and Ensembles

In this notebook we will demonstrate decision trees and ensembles like random forests, bagging, and boosting.

Links of interest:
- [Scikit-Learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- [Scikit-Learn: Bagging Meta Estimator](https://scikit-learn.org/stable/modules/ensemble.html#bagging-meta-estimator)
- [Scikit-Learn: Forests of Randomized Trees](https://scikit-learn.org/stable/modules/ensemble.html#forests-of-randomized-trees)
- [Scikit-Learn: Ensemble Methods: AdaBoost](https://scikit-learn.org/stable/modules/ensemble.html#adaboost)

# Imports

In [ ]:
import graphviz
from IPython.display import display_png
import numpy as np
import os
import pandas as pd
from pathlib import Path
import pydotplus
from sklearn.ensemble import BaggingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn import metrics

# We use two different plotting libraries, depending on which kind of plot we want
import matplotlib.pyplot as plt
import seaborn as sns

## Parameters

In [ ]:
# Set an option for Pandas to display smaller floating-point numbers
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
# Turn on Latex support for matplotlib (if you want it)
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Bookman"
})

In [ ]:
# Plotting values
background_color = "#fafafa"

In [ ]:
# Machine Epsilon, for computing division
eps = np.finfo(np.float32).eps

# Data Loading and Preprocessing

In [ ]:
from data_loading_functions import load_data

In [ ]:
data = load_data('../data/wisconsin_breast_cancer_data.csv')

# Split apart the pieces of the `data` dictionary
training_values = data['training_values']
training_labels = data['training_labels']
testing_values = data['testing_values']
testing_labels = data['testing_labels']

In [ ]:
feature_names = list(training_values.columns)

In [ ]:
testing_labels.head()

# Decision Tree Creation

In [ ]:
tree_clf = DecisionTreeClassifier()

# Fit (train) the classifier
tree_clf.fit(training_values, training_labels)

# Make predictions
tree_predictions = tree_clf.predict(testing_values) 

In [ ]:
print(55 * "=")
print("Decision Trees")
print(55 * "-")
print(metrics.classification_report(testing_labels, tree_predictions, target_names=['Benign', 'Malignant']))

In [ ]:
fig, ax = plt.subplots(dpi=100, facecolor=background_color)

ax.matshow(np.array([[1,0],[0,1]]), cmap=plt.cm.Blues, alpha=0.5)
ax.set_xticks([0, 1], ['Positive', 'Negative'])
ax.set_yticks([0, 1], ['Positive', 'Negative'])
ax.set_title('Actual vs. Predicted Classes')
ax.set_xlabel('Classifier Output')
ax.set_ylabel('Ground Truth')

ax.text(x=0, y=0, s=r'True\\Positive', va='center', ha='center', size='xx-large')
ax.text(x=1, y=1, s=r'True\\Negative', va='center', ha='center', size='xx-large')
ax.text(x=0, y=1, s=r'False\\Negative', va='center', ha='center', size='xx-large')
ax.text(x=1, y=0, s=r'False\\Positive', va='center', ha='center', size='xx-large')

plt.tight_layout()

In [ ]:
# Calculate the Confusion Matrix
tree_matrix = metrics.confusion_matrix(testing_labels, tree_predictions)

# Print confusion matrix
print(55 * "=")
print("Decision Trees")
print(55 * "-")

print("True Positive: {}".format(tree_matrix[1][1]))
print("True Negative: {}".format(tree_matrix[0][0]))
print("False Positive: {}".format(tree_matrix[0][1]))
print("False Negative: {}".format(tree_matrix[1][0]))


## Better Confusion Matrix

In [ ]:
def draw_conf_mat(y_true, y_prediction):
    conf_mat = confusion_matrix(y_true, y_prediction)
    
    fig, ax = plt.subplots(dpi=100, figsize=(5,5), facecolor=background_color)

    ax.matshow(conf_mat, cmap=plt.cm.Blues, alpha=0.5)
    ax.set_xticks([0, 1], ['Benign', 'Malignant'])
    ax.set_yticks([0, 1], ['Benign', 'Malignant'])
    ax.set_title('Actual vs. Predicted Malignancy')
    ax.set_xlabel('Classifier Output')
    ax.set_ylabel('Ground Truth')
    
    for i in range(conf_mat.shape[0]):
        for j in range(conf_mat.shape[1]):
            ax.text(x=j, y=i,s=conf_mat[i, j], va='center', ha='center', size='xx-large')
    plt.tight_layout()

In [ ]:
draw_conf_mat(testing_labels, tree_predictions)

## Display Decision Tree Structure

In [ ]:
dot_data = tree.export_graphviz(tree_clf, out_file=None, 
                                feature_names=training_values.columns,  
                                class_names=["Benign", "Malignant"],  
                                filled=True, rounded=True,  
                                special_characters=True)  

# Create the graph from the dot data
pydot_graph = pydotplus.graph_from_dot_data(dot_data)

# Set the output size
pydot_graph.set_size('"20,20!"')
#pydot_graph.set_size('"5,5!"')

# Create the graphviz object
gvz_graph = graphviz.Source(pydot_graph.to_string())

In [ ]:
# View
display_png(gvz_graph)

# Sensitivity of Decision Trees to Training Data

In [ ]:
# Create a vector to randomly split the data
n_training = len(training_values)
idx_split = np.arange(0, n_training)
np.random.shuffle(idx_split)

training_values_a = training_values.iloc[idx_split[:int(n_training/2)], :]
training_labels_a = training_labels.iloc[idx_split[:int(n_training/2)]]

training_values_b = training_values.iloc[idx_split[int(n_training/2):], :]
training_labels_b = training_labels.iloc[idx_split[int(n_training/2):]]

In [ ]:
n_training

In [ ]:
# Create two different trees
tree_a = DecisionTreeClassifier()
tree_a.fit(training_values_a, training_labels_a)

tree_b = DecisionTreeClassifier()
tree_b.fit(training_values_b, training_labels_b)

In [ ]:
# Display tree A
dot_data_a = tree.export_graphviz(tree_a, out_file=None, 
                                feature_names=training_values.columns,  
                                class_names=["B", "M"],  
                                filled=True, rounded=True,  
                                special_characters=True)  

# Create the graph from the dot data
pydot_graph_a = pydotplus.graph_from_dot_data(dot_data_a)

# Set the output size
pydot_graph_a.set_size('"20,20!"')

# Create the graphviz object
gvz_graph_a = graphviz.Source(pydot_graph_a.to_string())

In [ ]:
# Display tree B
dot_data_b = tree.export_graphviz(tree_b, out_file=None, 
                                feature_names=training_values.columns,  
                                class_names=["B", "M"],
                                filled=True, rounded=True,  
                                special_characters=True)  

# Create the graph from the dot data
pydot_graph_b = pydotplus.graph_from_dot_data(dot_data_b)

# Set the output size
pydot_graph_b.set_size('"10,10!"')

# Create the graphviz object
gvz_graph_b = graphviz.Source(pydot_graph_b.to_string())

In [ ]:
display_png(gvz_graph_a)

In [ ]:
display_png(gvz_graph_b)

In [ ]:
predictions_a = tree_a.predict(testing_values)
predictions_b = tree_b.predict(testing_values)

In [ ]:
draw_conf_mat(testing_labels, predictions_a)
draw_conf_mat(testing_labels, predictions_b)

# Random Forests

In [ ]:
from sklearn.ensemble import RandomForestClassifier

max_features = 10
n_estimators = 5
 
rf_clf = RandomForestClassifier(max_features=max_features, 
                                n_estimators=n_estimators)
rf_clf.fit(training_values, np.array(training_labels).ravel())

# Make predictions
rf_predictions = rf_clf.predict(testing_values)

In [ ]:
print(55 * "=")
print(f"Random Forests with {max_features} features and {n_estimators} trees:")
print(55 * "-")
print(metrics.classification_report(testing_labels, rf_predictions, target_names=['Benign', 'Malignant']))

In [ ]:
draw_conf_mat(testing_labels, rf_predictions)

## Try Several Random Forest Settings

In [ ]:
for max_features in [1, 3, 5, 10, 20]:
  for n_estimators in [50]:
    rf_clf = RandomForestClassifier(max_features=max_features, n_estimators=n_estimators)
    rf_clf.fit(training_values, np.array(training_labels).ravel())

    # Make predictions
    rf_predictions = rf_clf.predict(testing_values)
    print(55 * "=")
    print(f"Random Forests with {max_features} features and {n_estimators} trees:")
    print(55 * "-")
    print(metrics.classification_report(testing_labels, rf_predictions, target_names=['Benign', 'Malignant']))

# Bagging Estimators

In [ ]:
# Template classifier object
tree_clf = DecisionTreeClassifier()

tree_bag = BaggingClassifier(tree_clf,
                             max_samples=0.5, 
                             max_features=0.5)

In [ ]:
# Train each of the classifiers on the training data
tree_clf.fit(training_values, np.array(training_labels).ravel())
tree_bag.fit(training_values, np.array(training_labels).ravel())


In [ ]:
# Get Predictions
tree_clf_predictions = tree_clf.predict(testing_values)
tree_bag_predictions = tree_bag.predict(testing_values)


In [ ]:
# Evaluate
print(55 * "=")
print("Decision Tree Classifier")
print(55 * "-")
print(metrics.classification_report(testing_labels, tree_clf_predictions, target_names=['Benign', 'Malignant']))

In [ ]:
print(55 * "=")
print("Decision Tree Bagging")
print(55 * "-")
print(metrics.classification_report(testing_labels, tree_bag_predictions, target_names=['Benign', 'Malignant']))

# Testing Robustness and Stability

It looks like bagging does better than vanilla decision trees, but is this result "stable"?

That is, if we randomly select a new bagged decision tree, will it still perform better?

In [ ]:
#  How many trials to run
n_repeat = 50

# Size of the training set to use in each trial
n_train = 200

In [ ]:
estimators = [("Decision Tree", DecisionTreeClassifier()),
              ("Bagging (Decision Tree)", BaggingClassifier(DecisionTreeClassifier()))]

n_estimators = len(estimators)

In [ ]:
# Loop over estimators to compare
estimator_scores = []
estimator_stdevs = []

for n, (name, estimator) in enumerate(estimators):

    # Compute predictions
    y_scores = []

    for i in range(n_repeat):
        training_idx = np.arange(0, len(training_labels))
        np.random.shuffle(training_idx)
        training_idx = training_idx[:n_train]
        
        estimator.fit(training_values.iloc[training_idx,:], np.array(training_labels.iloc[training_idx]).ravel())

        y_predict = estimator.predict(testing_values)
        y_scores.append(metrics.f1_score(testing_labels, y_predict))

    # Print the results
    print(f'F-1 Scores for {name}:')
    print(f'Average Score: {np.mean(y_scores):.2}')
    print(f'STD of Score: {np.std(y_scores):.2}')
    print()
    estimator_scores.append(np.mean(y_scores))
    estimator_stdevs.append(np.std(y_scores))

To summarize these results it's helpful to look at the performance visually.

In [ ]:
# Organize data for the plots
estimator_names = ['Decision Tree', 'Bagging (Decision Tree)']
x_pos = np.arange(len(estimator_names))

# Create the holder for the plots
f, ax = plt.subplots(figsize=(5,4), dpi=150, facecolor=background_color)
ax.set_facecolor(background_color)

ax.bar(x_pos, estimator_scores, yerr=estimator_stdevs, align='center', alpha=0.5, ecolor='black', capsize=10)

ax.set_xticks(x_pos)
ax.set_xticklabels(estimator_names)
ax.yaxis.grid(True, linestyle=':')
ax.set_ylim(0.0, 1.0)

ax.set(xlabel=r'Estimator Name',
       ylabel=r'Scores',
       title='Estimator Scores for Vanilla and Bagged Estimators',
       #ylim=(0, 1.0),
      )
plt.tight_layout()

plt.show()

# AdaBoost

In [ ]:
# Create and fit an AdaBoosted decision tree
from sklearn.ensemble import AdaBoostClassifier

bdt = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1),
                         n_estimators=1)

# Train using just two features so we can visualize
X_train = training_values#.iloc[:,:2]
y_train = training_labels#.iloc[:]

bdt.fit(X_train, np.array(y_train).ravel())


In [ ]:
bdt_predictions = bdt.predict(testing_values)

In [ ]:
print(55 * "=")
print("Decision Tree AdaBoost")
print(55 * "-")
print(metrics.classification_report(testing_labels, bdt_predictions, target_names=['Benign', 'Malignant']))

In [ ]:
draw_conf_mat(testing_labels, bdt_predictions)